In [188]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [189]:
%cd /content/drive/MyDrive/projects

/content/drive/MyDrive/projects


In [190]:
import pandas as pd
import numpy as np
import sklearn
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error
from statsmodels.tsa.stattools import kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import matplotlib.pyplot as plt

In [191]:
df = pd.read_csv("/content/drive/MyDrive/projects/09_03_Component_Performance.csv")

In [192]:
df_service = pd.read_csv("/content/drive/MyDrive/projects/09_06_Service_Intervals.csv")

In [193]:
df.head(10
        )

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model
0,VH-BDZ,hyd-214,asm_019,68,386.8,NaN,acc,OK,A320
1,VH-GMD,ENG-A01,ASM-001,145,136.0,04/08/2025,Secondary,OK,A350-900
2,VH-ERL,APU-03,ASM_030,432,618.1,06-Jun-25,Primary,OK,A321
3,VH-WHZ,strut-01,asm_025,93,NaN,08-Jul-24,Accessory,OK,B787-9
4,VH-BDZ,GEAR-MLG,asm-040,203,952.3,27/07/2025,Primary,OK,A320
5,VH-ARQ,gear_mlg,asm-040,107,489.5,01/08/2025,Secondary,OK,A320
6,VH-DHT,FLP_23B,ASM-013,83,189.5,04-29-24,Secondary,OK,A350-900
7,VH-ATD,pitot_02,ASM-050,89,240.3,26/02/2025,Primary,OK,A321
8,VH-BDY,nav-a7,ASM-020,64,140.8,08-25-25,Secondary,OK,A350-900
9,VH-CYG,gear_mlg,ASM-040,95,1078.5,24/10/2024,Secondary,Damaged,B787-9


In [194]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Aircraft_ID                 160 non-null    object 
 1   Component_ID                150 non-null    object 
 2   Assembly_ID                 160 non-null    object 
 3   Flights_Since_Service       160 non-null    int64  
 4   Flight_Hours_Since_Service  155 non-null    float64
 5   Last_Service_Date           150 non-null    object 
 6   Component_Grade             160 non-null    object 
 7   Condition_Status            160 non-null    object 
 8   Aircraft_Model              160 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 11.4+ KB


In [195]:
#component_ID different variation
df['Component_ID'].nunique()

37

In [196]:
for col in ['Component_ID', 'Assembly_ID']:
  print(df['Component_ID'].unique())

['hyd-214' 'ENG-A01' 'APU-03' 'strut-01' 'GEAR-MLG' 'gear_mlg' 'FLP_23B'
 'pitot_02' 'nav-a7' 'FUSL-11' nan 'CABN-05' 'STRUT-01' 'NAV-A7' 'fusl-11'
 'flp-23b' 'FLP-23B' 'pitot-02' 'nav_a7' 'NAV_A7' 'ENG-A-01' 'apu_03'
 'PITOT_02' 'PITOT-02' 'eng-a-01' 'strut_01' 'gear-mlg' 'eng-a01'
 'cabn-05' 'FUSL_11' 'apu-03' 'APU_03' 'eng_a01' 'HYD-214' 'GEAR_MLG'
 'ENG_A-01' 'CABN_05' 'ENG_A01']
['hyd-214' 'ENG-A01' 'APU-03' 'strut-01' 'GEAR-MLG' 'gear_mlg' 'FLP_23B'
 'pitot_02' 'nav-a7' 'FUSL-11' nan 'CABN-05' 'STRUT-01' 'NAV-A7' 'fusl-11'
 'flp-23b' 'FLP-23B' 'pitot-02' 'nav_a7' 'NAV_A7' 'ENG-A-01' 'apu_03'
 'PITOT_02' 'PITOT-02' 'eng-a-01' 'strut_01' 'gear-mlg' 'eng-a01'
 'cabn-05' 'FUSL_11' 'apu-03' 'APU_03' 'eng_a01' 'HYD-214' 'GEAR_MLG'
 'ENG_A-01' 'CABN_05' 'ENG_A01']


In [197]:
# fix Component_id by spelling
for col in ['Component_ID', 'Assembly_ID']:
    df[col] = df[col].astype(str).str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
def sanitize_ids(df, id_columns):
  for col in id_columns:
      df[col] = df[col].astype(str).str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
  return df


df = sanitize_ids(df, ['Component_ID', 'Assembly_ID'])
df = df.drop(columns=['Notes'], errors='ignore')

In [198]:
for col in ['Component_ID', 'Assembly_ID']:
  print(df['Component_ID'].unique())

['HYD214' 'ENGA01' 'APU03' 'STRUT01' 'GEARMLG' 'FLP23B' 'PITOT02' 'NAVA7'
 'FUSL11' 'NAN' 'CABN05']
['HYD214' 'ENGA01' 'APU03' 'STRUT01' 'GEARMLG' 'FLP23B' 'PITOT02' 'NAVA7'
 'FUSL11' 'NAN' 'CABN05']


In [199]:
# null component_id to cheack
df_null_Component_ID = df[df['Component_ID'] == 'NAN']

In [200]:
df_null_Component_ID

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model
11,VH-BDZ,NAN,ASM013,90,454.6,10/03/2024,Sec,OK,A320
15,VH-RGG,NAN,ASM025,163,774.0,08-Feb-25,Accessory,Damaged,B787-9
19,VH-MMD,NAN,ASM025,103,519.0,20-Jul-25,Secondary,Damaged,A320
28,VH-ALM,NAN,ASM013,144,275.9,24/06/2024,Primary,OK,A350-900
54,VH-CYG,NAN,ASM030,271,516.9,11/01/2024,Primary,OK,B787-9
55,VH-YGA,NAN,ASM013,77,205.7,04-24-25,Primary,Damaged,A320
66,VH-OLQ,NAN,ASM019,37,303.0,05-26-24,Accessory,OK,B737-800
105,VH-PFQ,NAN,ASM020,112,19.1,25/09/2024,Secondary,OK,A350-900
125,VH-ALM,NAN,ASM021,130,375.8,08/11/2024,Primary,OK,A350-900
132,VH-ERL,NAN,ASM030,181,691.4,15/12/2024,Primary,Damaged,A321


In [201]:
df = df[df['Component_ID'] != 'NAN']   # previous step converted  NAN into 'NAN'

In [202]:
# Verify the rows have been removed
df_null = df[df['Component_ID'] == 'NAN']
print(f"Number of rows with 'NAN' Component_ID after removal: {len(df_null)}")


Number of rows with 'NAN' Component_ID after removal: 0


## Component_Grade

In [203]:
df['Component_Grade'].unique()

array(['acc', 'Secondary', 'Primary', 'Accessory', 'PRIM.'], dtype=object)

In [204]:
# Manual Grouping Logic
grade_map = {
    'acc': 'Accessory',
    'Accessory': 'Accessory',
    'PRIM.': 'Primary',
    'Primary': 'Primary',
    'Secondary': 'Secondary'
}

df['Component_Grade'] = df['Component_Grade'].map(grade_map)

In [205]:
df['Component_Grade'].unique()

array(['Accessory', 'Secondary', 'Primary'], dtype=object)

In [206]:
def flexible_date_parser(date_str):
    if pd.isna(date_str) or date_str == 'NA':
        return None

    # List formats from most specific to most general
    formats = [
        '%d-%b-%Y', # 08-Apr-2025
        '%d/%m/%Y', # 07/11/2024
        '%Y-%m-%d', # 2024-12-01
        '%m-%d-%y'  # 07-11-24
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except (ValueError, TypeError):
            continue

    return pd.to_datetime(date_str, errors='coerce')


df['Last_Service_Date'] = df['Last_Service_Date'].apply(flexible_date_parser)

In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 159
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Aircraft_ID                 150 non-null    object        
 1   Component_ID                150 non-null    object        
 2   Assembly_ID                 150 non-null    object        
 3   Flights_Since_Service       150 non-null    int64         
 4   Flight_Hours_Since_Service  145 non-null    float64       
 5   Last_Service_Date           140 non-null    datetime64[ns]
 6   Component_Grade             150 non-null    object        
 7   Condition_Status            150 non-null    object        
 8   Aircraft_Model              150 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 11.7+ KB


In [208]:
df.head()

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model
0,VH-BDZ,HYD214,ASM019,68,386.8,NaT,Accessory,OK,A320
1,VH-GMD,ENGA01,ASM001,145,136.0,2025-08-04,Secondary,OK,A350-900
2,VH-ERL,APU03,ASM030,432,618.1,2025-06-06,Primary,OK,A321
3,VH-WHZ,STRUT01,ASM025,93,NaN,2024-07-08,Accessory,OK,B787-9
4,VH-BDZ,GEARMLG,ASM040,203,952.3,2025-07-27,Primary,OK,A320


## Service Intervals Pre-processing

In [209]:
df_service.head()

,Component_ID,Assembly_ID,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Notes,Last_Update
0,FUSL-11,asm-016,178,508,HIGH,Legacy spec,24-Oct-2023
1,pitot-02,ASM-050,118,250,High,Rev B,01-Sep-2023
2,NAV-A7,ASM-020,93,182,high,Per OEM SB-2022-14,2023-01-03
3,CABN-05,ASM021,117,299,HIGH,Legacy spec,2024-08-23
4,gear-mlg,asm-040,182,680,Low,Legacy spec,19-Aug-2024


In [210]:
def sanitize_ids(df, id_columns):
    for col in id_columns:
        df[col] = df[col].astype(str).str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
    return df

# Applying the same logic to the manufacturer data
df_service = sanitize_ids(df_service, ['Component_ID', 'Assembly_ID'])
df_service = df_service.drop(columns=['Notes'], errors='ignore')

In [211]:
df_service['Last_Update'] = df_service['Last_Update'].apply(flexible_date_parser)

##

In [212]:
latest_service_df = df_service.sort_values('Last_Update').drop_duplicates(
    subset=['Component_ID', 'Assembly_ID'],
    keep='last'
)

In [213]:
latest_service_df

,Component_ID,Assembly_ID,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update
4,GEARMLG,ASM040,182,680,Low,2024-08-19
77,STRUT01,ASM025,153,586,High,2024-08-23
117,CABN05,ASM021,123,317,Medium,2024-09-08
44,NAVA7,ASM020,90,191,high,2024-10-04
30,FUSL11,ASM016,182,493,Medium,2024-11-14
78,APU03,ASM030,204,644,Medium,2024-11-30
57,FLP23B,ASM013,84,182,HIGH,2024-12-07
19,ENGA01,ASM001,105,333,Medium,2024-12-11
120,HYD214,ASM019,61,220,Medium,2024-12-17
130,PITOT02,ASM050,123,249,high,2024-12-22


# The Multi-Key Join & Integrity Audit+

In [214]:
audit_merge = pd.merge(
    df,
    latest_service_df,
    on=['Component_ID', 'Assembly_ID'],
    how='left',
    indicator=True # This creates a '_merge' column telling us where the data came from
)

# 2. Isolate 'Investigation' records
investigation_needed = audit_merge[audit_merge['_merge'] == 'left_only']

# 3. Final Analysis Set (The 'Inner Join' equivalent)
audit_final = audit_merge[audit_merge['_merge'] == 'both'].drop(columns=['_merge'])

print(f"Audit set ready: {len(audit_final)} records.")
print(f"ATTENTION: {len(investigation_needed)} records missing manufacturer limits.")

Audit set ready: 144 records.
ATTENTION: 6 records missing manufacturer limits.


In [215]:
investigation_needed

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update,_merge
15,VH-YGA,HYD214,ASM013,65,153.0,2025-07-19,Accessory,OK,A320,NaN,NaN,NaN,NaT,left_only
59,VH-ATD,GEARMLG,ASM013,159,660.1,NaT,Primary,Damaged,A321,NaN,NaN,NaN,NaT,left_only
74,VH-MMD,ENGA01,ASM013,90,428.3,2025-04-11,Accessory,OK,A320,NaN,NaN,NaN,NaT,left_only
121,VH-YGA,CABN05,ASM013,153,208.1,2024-08-12,Primary,OK,A320,NaN,NaN,NaN,NaT,left_only
126,VH-PFQ,NAVA7,ASM013,100,107.4,2025-08-23,Primary,OK,A350-900,NaN,NaN,NaN,NaT,left_only
133,VH-MMD,CABN05,ASM013,86,201.5,2024-04-09,Accessory,OK,A320,NaN,NaN,NaN,NaT,left_only


In [216]:
audit_merge

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update,_merge
0,VH-BDZ,HYD214,ASM019,68,386.8,NaT,Accessory,OK,A320,61.0,220.0,Medium,2024-12-17,both
1,VH-GMD,ENGA01,ASM001,145,136.0,2025-08-04,Secondary,OK,A350-900,105.0,333.0,Medium,2024-12-11,both
2,VH-ERL,APU03,ASM030,432,618.1,2025-06-06,Primary,OK,A321,204.0,644.0,Medium,2024-11-30,both
3,VH-WHZ,STRUT01,ASM025,93,NaN,2024-07-08,Accessory,OK,B787-9,153.0,586.0,High,2024-08-23,both
4,VH-BDZ,GEARMLG,ASM040,203,952.3,2025-07-27,Primary,OK,A320,182.0,680.0,Low,2024-08-19,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,VH-RGG,ENGA01,ASM001,94,501.1,2025-05-17,Primary,OK,B787-9,105.0,333.0,Medium,2024-12-11,both
146,VH-BDZ,PITOT02,ASM050,39,284.5,2024-09-20,Primary,OK,A320,123.0,249.0,high,2024-12-22,both
147,VH-NBT,NAVA7,ASM020,81,258.6,2024-06-01,Primary,OK,A321,90.0,191.0,high,2024-10-04,both
148,VH-DHT,ENGA01,ASM001,73,324.1,NaT,Accessory,OK,A350-900,105.0,333.0,Medium,2024-12-11,both


In [217]:
audit_final['Is_Overdue'] = np.where(
    (audit_final['Flights_Since_Service'] > audit_final['Service_Flight_Limit']) |
    (audit_final['Flight_Hours_Since_Service'] > audit_final['Service_Hour_Limit']),
    True,
    False
)

In [218]:
audit_final

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update,Is_Overdue
0,VH-BDZ,HYD214,ASM019,68,386.8,NaT,Accessory,OK,A320,61.0,220.0,Medium,2024-12-17,True
1,VH-GMD,ENGA01,ASM001,145,136.0,2025-08-04,Secondary,OK,A350-900,105.0,333.0,Medium,2024-12-11,True
2,VH-ERL,APU03,ASM030,432,618.1,2025-06-06,Primary,OK,A321,204.0,644.0,Medium,2024-11-30,True
3,VH-WHZ,STRUT01,ASM025,93,NaN,2024-07-08,Accessory,OK,B787-9,153.0,586.0,High,2024-08-23,False
4,VH-BDZ,GEARMLG,ASM040,203,952.3,2025-07-27,Primary,OK,A320,182.0,680.0,Low,2024-08-19,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,VH-RGG,ENGA01,ASM001,94,501.1,2025-05-17,Primary,OK,B787-9,105.0,333.0,Medium,2024-12-11,True
146,VH-BDZ,PITOT02,ASM050,39,284.5,2024-09-20,Primary,OK,A320,123.0,249.0,high,2024-12-22,True
147,VH-NBT,NAVA7,ASM020,81,258.6,2024-06-01,Primary,OK,A321,90.0,191.0,high,2024-10-04,True
148,VH-DHT,ENGA01,ASM001,73,324.1,NaT,Accessory,OK,A350-900,105.0,333.0,Medium,2024-12-11,False


In [219]:
# Filter for only overdue components first
overdue_df = audit_final[audit_final['Is_Overdue'] == True].copy()

# Ensure Criticality is clean/uppercase
overdue_df['Criticality_Level'] = overdue_df['Criticality_Level'].str.upper()

# Pivot 1: Component Grade
p1 = overdue_df.pivot_table(index=['Aircraft_ID', 'Aircraft_Model'],
                            columns='Component_Grade', values='Component_ID', aggfunc='count').add_prefix('Comp - ')

# Pivot 2: Condition Status
p2 = overdue_df.pivot_table(index=['Aircraft_ID', 'Aircraft_Model'],
                            columns='Condition_Status', values='Component_ID', aggfunc='count').add_prefix('Condition - ')

# Pivot 3: Criticality Level
p3 = overdue_df.pivot_table(index=['Aircraft_ID', 'Aircraft_Model'],
                            columns='Criticality_Level', values='Component_ID', aggfunc='count').add_prefix('Crit - ')

# Join them all together on Aircraft_ID and Model
priority_df = p3.join([p2, p1], how='outer').fillna(0).reset_index()


In [220]:
priority_df

,Aircraft_ID,Aircraft_Model,Crit - HIGH,Crit - LOW,Crit - MEDIUM,Condition - Damaged,Condition - OK,Comp - Accessory,Comp - Primary,Comp - Secondary
0,VH-ALM,A350-900,0.0,2.0,1.0,2.0,1.0,1.0,0.0,2.0
1,VH-ARQ,A320,1.0,0.0,3.0,2.0,2.0,0.0,3.0,1.0
2,VH-ATD,A321,2.0,0.0,1.0,1.0,2.0,0.0,2.0,1.0
3,VH-BDY,A350-900,3.0,0.0,4.0,2.0,5.0,3.0,4.0,0.0
4,VH-BDZ,A320,3.0,1.0,5.0,0.0,9.0,2.0,6.0,1.0
5,VH-CSO,A320,1.0,0.0,4.0,1.0,4.0,0.0,2.0,3.0
6,VH-CYG,B787-9,3.0,1.0,1.0,1.0,4.0,0.0,4.0,1.0
7,VH-DHT,A350-900,1.0,0.0,2.0,0.0,3.0,1.0,0.0,2.0
8,VH-ERL,A321,1.0,1.0,5.0,1.0,6.0,1.0,4.0,2.0
9,VH-GMD,A350-900,2.0,0.0,4.0,1.0,5.0,0.0,3.0,3.0


In [221]:
# Priority 1 Score: High Criticality + Damaged Condition + Primary Grade
priority_df['Priority_1_Score'] = (
    priority_df.get('Crit - HIGH', 0) +
    priority_df.get('Condition - Damaged', 0) +
    priority_df.get('Comp - Primary', 0)
)

# Priority 2 Score: Medium Criticality + Secondary Grade
priority_df['Priority_2_Score'] = (
    priority_df.get('Crit - MEDIUM', 0) +
    priority_df.get('Comp - Secondary', 0)
)

# Sort by the most critical aircraft first
final_priority_list = priority_df.sort_values(by=['Priority_1_Score', 'Priority_2_Score'], ascending=False)

In [222]:
final_priority_list

,Aircraft_ID,Aircraft_Model,Crit - HIGH,Crit - LOW,Crit - MEDIUM,Condition - Damaged,Condition - OK,Comp - Accessory,Comp - Primary,Comp - Secondary,Priority_1_Score,Priority_2_Score
15,VH-RGG,B787-9,3.0,2.0,4.0,2.0,7.0,1.0,5.0,3.0,10.0,7.0
14,VH-RCU,A320,4.0,1.0,3.0,3.0,5.0,2.0,3.0,3.0,10.0,6.0
4,VH-BDZ,A320,3.0,1.0,5.0,0.0,9.0,2.0,6.0,1.0,9.0,6.0
3,VH-BDY,A350-900,3.0,0.0,4.0,2.0,5.0,3.0,4.0,0.0,9.0,4.0
19,VH-ZUM,A320,3.0,1.0,2.0,1.0,5.0,1.0,4.0,1.0,8.0,3.0
6,VH-CYG,B787-9,3.0,1.0,1.0,1.0,4.0,0.0,4.0,1.0,8.0,2.0
8,VH-ERL,A321,1.0,1.0,5.0,1.0,6.0,1.0,4.0,2.0,6.0,7.0
9,VH-GMD,A350-900,2.0,0.0,4.0,1.0,5.0,0.0,3.0,3.0,6.0,7.0
1,VH-ARQ,A320,1.0,0.0,3.0,2.0,2.0,0.0,3.0,1.0,6.0,4.0
2,VH-ATD,A321,2.0,0.0,1.0,1.0,2.0,0.0,2.0,1.0,5.0,2.0


##Maintenance Forecast

In [228]:
# Isolate the 'Safe' fleet (False for overdue)
upcoming_maintenance = audit_final[audit_final['Is_Overdue'] == False].copy()

# We create new boolean columns for each service window
for limit in [0.6, 0.8]:
    col_name = f'Service_Limit_{int(limit*100)}pct'

    # Formula: (Current Flights > 80% of Max) OR (Current Hours > 80% of Max)
    upcoming_maintenance[col_name] = (
        (upcoming_maintenance['Flights_Since_Service'] > (upcoming_maintenance['Service_Flight_Limit'] * limit)) |
        (upcoming_maintenance['Flight_Hours_Since_Service'] > (upcoming_maintenance['Service_Hour_Limit'] * limit))
    )

In [236]:
upcoming_maintenance.head()

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update,Is_Overdue,Service_Limit_60pct,Service_Limit_80pct,Maintenance_Tier
3,VH-WHZ,STRUT01,ASM025,93,NaN,2024-07-08,Accessory,OK,B787-9,153.0,586.0,High,2024-08-23,False,True,False,CAUTION: Within 40% of Limit
5,VH-ARQ,GEARMLG,ASM040,107,489.5,2025-08-01,Secondary,OK,A320,182.0,680.0,Low,2024-08-19,False,True,False,CAUTION: Within 40% of Limit
7,VH-ATD,PITOT02,ASM050,89,240.3,2025-02-26,Primary,OK,A321,123.0,249.0,high,2024-12-22,False,True,True,CRITICAL: Within 20% of Limit
8,VH-BDY,NAVA7,ASM020,64,140.8,2025-08-25,Secondary,OK,A350-900,90.0,191.0,high,2024-10-04,False,True,False,CAUTION: Within 40% of Limit
14,VH-YGA,NAVA7,ASM020,58,176.2,2024-06-22,Secondary,OK,A320,90.0,191.0,high,2024-10-04,False,True,True,CRITICAL: Within 20% of Limit


In [231]:
def assign_priority_tier(row):
    if row['Service_Limit_80pct']: return 'CRITICAL: Within 20% of Limit'
    if row['Service_Limit_60pct']: return 'CAUTION: Within 40% of Limit'
    return 'SAFE'

upcoming_maintenance['Maintenance_Tier'] = upcoming_maintenance.apply(assign_priority_tier, axis=1)

# Keep only the rows that need monitoring
forecast_report = upcoming_maintenance[upcoming_maintenance['Maintenance_Tier'] != 'SAFE']

In [235]:
forecast_report

,Aircraft_ID,Component_ID,Assembly_ID,Flights_Since_Service,Flight_Hours_Since_Service,Last_Service_Date,Component_Grade,Condition_Status,Aircraft_Model,Service_Flight_Limit,Service_Hour_Limit,Criticality_Level,Last_Update,Is_Overdue,Service_Limit_60pct,Service_Limit_80pct,Maintenance_Tier
3,VH-WHZ,STRUT01,ASM025,93,NaN,2024-07-08,Accessory,OK,B787-9,153.0,586.0,High,2024-08-23,False,True,False,CAUTION: Within 40% of Limit
5,VH-ARQ,GEARMLG,ASM040,107,489.5,2025-08-01,Secondary,OK,A320,182.0,680.0,Low,2024-08-19,False,True,False,CAUTION: Within 40% of Limit
7,VH-ATD,PITOT02,ASM050,89,240.3,2025-02-26,Primary,OK,A321,123.0,249.0,high,2024-12-22,False,True,True,CRITICAL: Within 20% of Limit
8,VH-BDY,NAVA7,ASM020,64,140.8,2025-08-25,Secondary,OK,A350-900,90.0,191.0,high,2024-10-04,False,True,False,CAUTION: Within 40% of Limit
14,VH-YGA,NAVA7,ASM020,58,176.2,2024-06-22,Secondary,OK,A320,90.0,191.0,high,2024-10-04,False,True,True,CRITICAL: Within 20% of Limit
17,VH-ERL,FUSL11,ASM016,141,296.0,2025-01-19,Primary,OK,A321,182.0,493.0,Medium,2024-11-14,False,True,False,CAUTION: Within 40% of Limit
19,VH-DHT,FLP23B,ASM013,77,94.4,2025-02-04,Primary,OK,A350-900,84.0,182.0,HIGH,2024-12-07,False,True,True,CRITICAL: Within 20% of Limit
20,VH-ERL,FLP23B,ASM013,61,86.6,2025-08-22,Secondary,Damaged,A321,84.0,182.0,HIGH,2024-12-07,False,True,False,CAUTION: Within 40% of Limit
21,VH-ATD,PITOT02,ASM050,123,116.0,2024-10-20,Secondary,OK,A321,123.0,249.0,high,2024-12-22,False,True,True,CRITICAL: Within 20% of Limit
24,VH-BDZ,NAVA7,ASM020,45,184.2,2025-04-08,Primary,OK,A320,90.0,191.0,high,2024-10-04,False,True,True,CRITICAL: Within 20% of Limit
